# NAC vs MCP: Token Efficiency Benchmark

Controlled comparison of Notion Agent CLI (NAC, 47 compound actions, markdown I/O) vs Notion MCP (endpoint-level tools, JSON I/O) across 8 task scenarios.

In [149]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
COMPARE_RUNS = {
    "Notion Agent CLI (Sonnet)":  ("20260316-191644", "nac"),
    "Notion MCP (Sonnet)":       ("20260316-191644", "mcp"),
    "Notion Agent CLI (Opus)":   ("20260316-214930", "nac"),
    "Notion MCP (Opus)":         ("20260316-214930", "mcp"),
}
# Set COMPARE_RUNS = None to use auto-detect (picks latest nac + mcp dirs)
SELECTED_RUNS = "latest"
MIN_TURNS = 2

# ── Imports ────────────────────────────────────────────────────────────────────
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Shared chart config ───────────────────────────────────────────────────────
PALETTE = {
    "Notion Agent CLI (Sonnet)": "#2563EB", "Notion MCP (Sonnet)": "#DC2626",
    "Notion Agent CLI (Opus)":   "#7C3AED", "Notion MCP (Opus)":   "#EA580C",
    "nac": "#2563EB", "mcp": "#DC2626",
}
SCENARIO_NAMES = {
    1: "Read+Summarize", 2: "Query+Report", 3: "Copy+Modify", 4: "Set Property",
    5: "Replace Section", 6: "Merge Pages", 7: "Batch Update", 8: "Copy+Append",
}
SHORT_NAMES = {1: "S1", 2: "S2", 3: "S3", 4: "S4", 5: "S5", 6: "S6", 7: "S7", 8: "S8"}
LAYOUT = dict(template="plotly_white", font_size=13)
LEGEND_TOP = dict(orientation="h", yanchor="bottom", y=1.08, x=0.5, xanchor="center")

RESULTS_BASE = Path("results") if Path("results").exists() else (
    Path("../benchmark/results") if Path("../benchmark/results").exists() else Path("benchmark/results"))
ASSETS_DIR = RESULTS_BASE.parent.parent / "assets"

In [150]:
# ── Environment metadata ──────────────────────────────────────────────────────
if COMPARE_RUNS:
    _rids = list({v[0] if isinstance(v, (list, tuple)) else v for v in COMPARE_RUNS.values()})
else:
    _rids = SELECTED_RUNS if isinstance(SELECTED_RUNS, list) else []

for rid in _rids:
    p = RESULTS_BASE / rid / "env.json"
    if p.exists():
        e = json.loads(p.read_text())
        print(f"{rid}: model={e.get('model')}, cli={e.get('claude_cli_version')}")

20260316-191644: model=claude-sonnet-4-6, cli=2.1.77 (Claude Code)
20260316-214930: model=claude-opus-4-6, cli=2.1.76 (Claude Code)


## 1. Data Loading

In [151]:
def parse_summary(path: Path) -> dict | None:
    try:
        data = json.loads(path.read_text())
    except (json.JSONDecodeError, OSError):
        return None
    result = str(data.get("result", ""))
    return {
        "num_turns": data.get("num_turns", 0),
        "cost_usd": data.get("total_cost_usd", data.get("cost_usd", 0)),
        "is_error": any(s in result for s in ["hit your limit", "rate_limit"]),
        "duration_ms": data.get("total_duration_ms", 0),
    }


def parse_jsonl(path: Path) -> dict:
    try:
        lines = [json.loads(l) for l in path.read_text().strip().splitlines() if l.strip()]
    except (json.JSONDecodeError, OSError):
        return {}
    if lines and lines[0].get("error") == "jsonl_missing":
        return {"tools_used": "missing"}

    total_input = total_output = total_cache_read = total_cache_creation = 0
    has_bash = has_mcp = False
    prev_sig = None
    turns = []

    for r in lines:
        if r.get("type") != "assistant":
            continue
        usage = r.get("message", {}).get("usage", {})
        if not usage:
            continue
        inp = usage.get("input_tokens", 0)
        cr = usage.get("cache_read_input_tokens", 0)
        cc = usage.get("cache_creation_input_tokens", 0)
        out = usage.get("output_tokens", 0)
        sig = (inp, cr, cc)
        is_new = sig != prev_sig
        prev_sig = sig

        if is_new:
            total_input += inp; total_cache_read += cr; total_cache_creation += cc
            turns.append({"input": inp, "cache_read": cr, "cache_create": cc,
                          "output": out, "billed": inp + cc})
        else:
            if turns:
                turns[-1]["output"] += out
        total_output += out

        for item in r.get("message", {}).get("content", []):
            if isinstance(item, dict) and item.get("type") == "tool_use":
                name = item.get("name", "")
                if name == "Bash": has_bash = True
                elif "__" in name: has_mcp = True

    tools_used = ("mixed" if has_bash and has_mcp else "cli" if has_bash else "mcp" if has_mcp else "none")
    return {"total_input": total_input, "total_output": total_output,
            "total_cache_read": total_cache_read, "total_cache_creation": total_cache_creation,
            "billed_input": total_input + total_cache_creation, "tools_used": tools_used,
            "turn_tokens": turns}


def load_run(run_id: str, label_override=None, file_prefix=None):
    run_dir = RESULTS_BASE / run_id
    sessions = []
    for f in sorted(run_dir.glob("*.json")):
        if f.name in ("env.json", "behavior.json"): continue
        if file_prefix and not f.stem.startswith(file_prefix + "-"): continue
        parts = f.stem.split("-")
        if len(parts) < 3: continue
        s = parse_summary(f)
        if not s or s["is_error"] or s["num_turns"] < MIN_TURNS: continue
        jd = parse_jsonl(f.with_suffix(".jsonl")) if f.with_suffix(".jsonl").exists() else {}
        plugin = parts[0]
        tu = jd.get("tools_used", "unknown")
        valid = tu not in ("mixed", "missing")
        if plugin == "mcp" and tu == "cli": valid = False
        if plugin == "nac" and tu == "mcp": valid = False
        snum = int(parts[1][1:])
        sessions.append({"run_id": run_id, "name": f.stem, "plugin": plugin,
            "label": label_override or plugin,
            "scenario": f"S{snum} {SCENARIO_NAMES.get(snum, '')}",
            "scenario_short": SHORT_NAMES.get(snum, f"S{snum}"),
            "scenario_num": snum, "iteration": int(parts[2]),
            "num_turns": s["num_turns"], "cost_usd": s["cost_usd"],
            "duration_s": s["duration_ms"] / 1000,
            "tools_used": tu, "valid": valid,
            **{k: v for k, v in jd.items() if k not in ("tools_used", "turn_tokens")},
            "turn_tokens": jd.get("turn_tokens", [])})
    return pd.DataFrame(sessions)


# ── Build DataFrame ────────────────────────────────────────────────────────────
if COMPARE_RUNS:
    dfs = []
    for label, val in COMPARE_RUNS.items():
        rid, prefix = (val if isinstance(val, (list, tuple)) else (val, None))
        rdf = load_run(rid, label_override=label, file_prefix=prefix)
        if not rdf.empty: dfs.append(rdf)
    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    labels = list(COMPARE_RUNS.keys())
else:
    all_runs = [d.name for d in sorted(RESULTS_BASE.iterdir()) if d.is_dir() and not d.name.startswith("_")]
    if SELECTED_RUNS == "latest":
        selected, has_nac, has_mcp = [], False, False
        for rid in reversed(all_runs):
            d = RESULTS_BASE / rid
            if any(d.glob("nac-*.json")) and not has_nac:
                has_nac = True; selected.append(rid)
            if any(d.glob("mcp-*.json")) and not has_mcp:
                has_mcp = True
                if rid not in selected: selected.append(rid)
            if has_nac and has_mcp: break
    else:
        selected = SELECTED_RUNS if isinstance(SELECTED_RUNS, list) else [SELECTED_RUNS]
    dfs = [load_run(rid) for rid in selected]
    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    df["label"] = df["plugin"]
    labels = sorted(df.label.unique().tolist())

if "valid" in df.columns:
    n_bad = (~df.valid).sum()
    if n_bad: print(f"Excluded {n_bad} contaminated sessions")
    df = df[df.valid].copy()

run_label = " vs ".join(labels)
print(f"{run_label}: {len(df)} sessions, scenarios {sorted(df.scenario_num.unique())}")

Notion Agent CLI (Sonnet) vs Notion MCP (Sonnet) vs Notion Agent CLI (Opus) vs Notion MCP (Opus): 160 sessions, scenarios [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


## 2. Summary

In [152]:
# Build summary with plotly table for clean rendering
header_vals = ["Scenario"]
for label in labels:
    header_vals += [f"{label} turns", f"{label} cost"]
if len(labels) == 2:
    header_vals += ["Turn \u0394", "Cost \u0394"]

col_data = {h: [] for h in header_vals}

for s in sorted(df.scenario_num.unique()):
    col_data["Scenario"].append(f"S{s} {SCENARIO_NAMES.get(s, '')}")
    vals = {}
    for label in labels:
        sub = df[(df.scenario_num == s) & (df.label == label)]
        if sub.empty:
            col_data[f"{label} turns"].append("-")
            col_data[f"{label} cost"].append("-")
            vals[label] = (0, 0)
            continue
        t_m, t_s = sub.num_turns.mean(), sub.num_turns.std()
        c_m = sub.cost_usd.mean()
        std = f" \u00b1{t_s:.1f}" if pd.notna(t_s) and t_s > 0.01 else ""
        col_data[f"{label} turns"].append(f"{t_m:.1f}{std}")
        col_data[f"{label} cost"].append(f"${c_m:.4f}")
        vals[label] = (t_m, c_m)
    if len(labels) == 2:
        la, lb = labels
        at, ac = vals.get(la, (0, 0))
        bt, bc = vals.get(lb, (0, 0))
        col_data["Turn \u0394"].append(f"{(1-at/bt)*100:+.0f}%" if bt else "-")
        col_data["Cost \u0394"].append(f"{(1-ac/bc)*100:+.0f}%" if bc else "-")

# Totals
col_data["Scenario"].append("<b>TOTAL</b>")
for label in labels:
    sub = df[df.label == label]
    col_data[f"{label} turns"].append(f"<b>{sub.num_turns.mean():.1f} avg</b>")
    col_data[f"{label} cost"].append(f"<b>${sub.cost_usd.sum():.4f}</b>")
if len(labels) == 2:
    la, lb = labels
    a_t, b_t = df[df.label == la].cost_usd.sum(), df[df.label == lb].cost_usd.sum()
    col_data["Turn \u0394"].append("")
    col_data["Cost \u0394"].append(f"<b>{(1-a_t/b_t)*100:+.0f}%</b>" if b_t else "-")

# Color the delta columns
def delta_colors(vals):
    return ["#dcfce7" if v.startswith("+") else "#fef2f2" if v.startswith("-") else "white" for v in vals]

fill_colors = []
for h in header_vals:
    if "\u0394" in h:
        fill_colors.append(delta_colors(col_data[h]))
    else:
        fill_colors.append(["white"] * len(col_data[h]))

fig = go.Figure(go.Table(
    header=dict(values=header_vals, fill_color="#f8fafc", font=dict(size=11),
                align=["left"] + ["right"] * (len(header_vals) - 1), line_color="#e2e8f0"),
    cells=dict(values=[col_data[h] for h in header_vals], fill_color=fill_colors,
               font=dict(size=11), align=["left"] + ["right"] * (len(header_vals) - 1),
               line_color="#e2e8f0", height=28),
))
fig.update_layout(title=run_label, height=60 + 28 * (len(col_data["Scenario"]) + 1),
                  margin=dict(t=40, b=10, l=10, r=10))
fig.show()

## 3. Mean Session Cost by Scenario

Horizontal bars show mean cost per session (USD). Error bars are ± 1 SD. n=5 iterations per condition per scenario. Both panels share the same x-axis for cross-model comparison.

In [153]:
def _model_pairs(labels):
    pairs = []
    for model in ("Sonnet", "Opus"):
        cli = next((l for l in labels if model in l and "CLI" in l), None)
        mcp = next((l for l in labels if model in l and "MCP" in l), None)
        if cli and mcp:
            pairs.append((model, cli, mcp))
    return pairs

NAC_COLOR = "#2563EB"
MCP_COLOR = "#DC2626"

pairs = _model_pairs(labels)
agg = df.groupby(["scenario_num", "label"]).agg(
    cost_mean=("cost_usd", "mean"), cost_std=("cost_usd", "std"),
    n=("cost_usd", "count")).reset_index()
scenario_labels = [f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in sorted(agg.scenario_num.unique())]
x_max = (agg.cost_mean + agg.cost_std.fillna(0)).max() * 1.30

fig = make_subplots(rows=1, cols=len(pairs),
    subplot_titles=["Sonnet 4.6", "Opus 4.6"][:len(pairs)],
    horizontal_spacing=0.20, shared_yaxes=True)

for col, (model, cli_label, mcp_label) in enumerate(pairs, 1):
    for label, color, name in [(mcp_label, MCP_COLOR, "MCP"), (cli_label, NAC_COLOR, "NAC")]:
        sub = agg[agg.label == label].sort_values("scenario_num")
        sub_labels = sub.scenario_num.map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")

        # Only label MCP bars (longer side) to reduce clutter
        texts = [f"${r.cost_mean:.3f}" for _, r in sub.iterrows()] if name == "MCP" else [""] * len(sub)

        fig.add_trace(go.Bar(
            y=sub_labels, x=sub.cost_mean, orientation="h",
            error_x=dict(type="data", array=sub.cost_std.fillna(0), thickness=1.5, width=3),
            name=name, legendgroup=name, showlegend=(col == 1),
            marker_color=color, opacity=0.85,
            text=texts, textposition="outside", textfont_size=10,
            hovertemplate="%{y}<br>" + name + ": $%{x:.4f} \u00b1 $%{error_x.array:.4f}<extra></extra>",
        ), row=1, col=col)

    fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(scenario_labels)), row=1, col=col)
    fig.update_xaxes(title_text="USD", tickprefix="$", range=[0, x_max], row=1, col=col)

fig.update_layout(**LAYOUT, barmode="group", height=540, width=1100,
    title=dict(text="Mean Session Cost by Scenario (\u00b1 SD)",
               y=0.98, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=100, b=50, l=160, r=80),
    legend=dict(orientation="h", yanchor="bottom", y=1.04, x=0.5, xanchor="center"))
fig.show()

## 3b. Paired Comparison (Turns + Cost by Model)

In [154]:
agg_paired = df.groupby(["scenario_num", "scenario_short", "label"]).agg(
    turns_mean=("num_turns", "mean"), cost_mean=("cost_usd", "mean")).reset_index()
scenario_labels_full = [f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in sorted(agg_paired.scenario_num.unique())]

titles = [f"Turns — {m}" for m, _, _ in pairs] + [f"Cost — {m}" for m, _, _ in pairs]
fig = make_subplots(rows=2, cols=len(pairs), subplot_titles=titles,
    horizontal_spacing=0.16, vertical_spacing=0.16)

turns_max = float(agg_paired.turns_mean.max()) * 1.20
cost_max = float(agg_paired.cost_mean.max()) * 1.20

for col, (model, cli_label, mcp_label) in enumerate(pairs, 1):
    cli = agg_paired[agg_paired.label == cli_label][["scenario_num", "turns_mean", "cost_mean"]].rename(
        columns={"turns_mean": "turns_cli", "cost_mean": "cost_cli"})
    mcp = agg_paired[agg_paired.label == mcp_label][["scenario_num", "turns_mean", "cost_mean"]].rename(
        columns={"turns_mean": "turns_mcp", "cost_mean": "cost_mcp"})
    merged = cli.merge(mcp, on="scenario_num", how="inner").sort_values("scenario_num")
    merged["scenario_label"] = merged["scenario_num"].map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")

    for row, (cli_m, mcp_m, x_title, x_max, fmt) in enumerate([
        ("turns_cli", "turns_mcp", "Turns", turns_max, "%{x:.1f} turns"),
        ("cost_cli", "cost_mcp", "USD", cost_max, "$%{x:.3f}"),
    ], 1):
        for _, item in merged.iterrows():
            lo, hi = sorted((item[cli_m], item[mcp_m]))
            fig.add_trace(go.Scatter(x=[lo, hi], y=[item["scenario_label"], item["scenario_label"]],
                mode="lines", line=dict(color="#CBD5E1", width=3),
                showlegend=False, hoverinfo="skip"), row=row, col=col)
        fig.add_trace(go.Scatter(x=merged[cli_m], y=merged["scenario_label"], mode="markers",
            name="NAC", legendgroup="NAC", showlegend=(col == 1 and row == 1),
            marker=dict(color=NAC_COLOR, size=11, line=dict(color="white", width=1)),
            hovertemplate="%{y}<br>NAC: " + fmt + "<extra>" + model + "</extra>"),
            row=row, col=col)
        fig.add_trace(go.Scatter(x=merged[mcp_m], y=merged["scenario_label"], mode="markers",
            name="MCP", legendgroup="MCP", showlegend=(col == 1 and row == 1),
            marker=dict(color=MCP_COLOR, size=11, line=dict(color="white", width=1)),
            hovertemplate="%{y}<br>MCP: " + fmt + "<extra>" + model + "</extra>"),
            row=row, col=col)

        fig.update_xaxes(title_text=x_title, range=[0, x_max], row=row, col=col)
        if row == 2:
            fig.update_xaxes(tickprefix="$", row=row, col=col)
        fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(scenario_labels_full)),
            showticklabels=(col == 1), row=row, col=col)

fig.update_layout(**LAYOUT, height=760, width=1180,
    title=dict(text="Per-Scenario Comparison: NAC vs MCP by Model",
               y=0.985, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=120, b=60, l=170, r=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0.5, xanchor="center"))
fig.show()

## 3c. Individual Sessions (raw points)

Each dot is one session. Bars show the mean. With n=5, every data point is visible, making outliers and clustering patterns transparent.

In [155]:
agg_pts = df.groupby(["scenario_num", "label"]).agg(
    cost_mean=("cost_usd", "mean")).reset_index()
scenario_labels_pts = [f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in sorted(agg_pts.scenario_num.unique())]
x_max_pts = df.cost_usd.max() * 1.15

fig = make_subplots(rows=1, cols=len(pairs),
    subplot_titles=["Sonnet 4.6", "Opus 4.6"][:len(pairs)],
    horizontal_spacing=0.20, shared_yaxes=True)

for col, (model, cli_label, mcp_label) in enumerate(pairs, 1):
    for label, color, name in [(mcp_label, MCP_COLOR, "MCP"), (cli_label, NAC_COLOR, "NAC")]:
        # Mean bars (thin, semi-transparent)
        sub_agg = agg_pts[agg_pts.label == label].sort_values("scenario_num")
        sub_labels = sub_agg.scenario_num.map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")
        fig.add_trace(go.Bar(
            y=sub_labels, x=sub_agg.cost_mean, orientation="h",
            name=name + " mean", legendgroup=name, showlegend=False,
            marker_color=color, opacity=0.25, width=0.35,
            hoverinfo="skip",
        ), row=1, col=col)

        # Raw session points (jittered vertically)
        sub_raw = df[df.label == label].sort_values(["scenario_num", "iteration"])
        raw_labels = sub_raw.scenario_num.map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")

        fig.add_trace(go.Scatter(
            x=sub_raw.cost_usd, y=raw_labels, mode="markers",
            name=name, legendgroup=name, showlegend=(col == 1),
            marker=dict(color=color, size=8, opacity=0.85,
                        line=dict(color="white", width=0.5)),
            hovertemplate="%{y}<br>" + name + ": $%{x:.4f}<br>%{text}<extra></extra>",
            text=sub_raw.name,
        ), row=1, col=col)

    fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(scenario_labels_pts)),
        row=1, col=col)
    fig.update_xaxes(title_text="USD", tickprefix="$", range=[0, x_max_pts], row=1, col=col)

fig.update_layout(**LAYOUT, barmode="overlay", height=540, width=1100,
    title=dict(text="Session Cost: Individual Observations",
               y=0.98, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=100, b=50, l=160, r=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.04, x=0.5, xanchor="center"))
fig.show()

## 4. Session Variability

In [156]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Turns per Session", "Cost per Session"],
                    horizontal_spacing=0.14)

for label in labels:
    sub = df[df.label == label].sort_values("scenario_num")
    color = PALETTE.get(label, "#888")

    fig.add_trace(go.Box(
        x=sub.scenario_short, y=sub.num_turns, name=label, legendgroup=label,
        marker_color=color, boxpoints="all", jitter=0.4, pointpos=0,
        marker_size=6, line_width=1.5,
        hovertemplate="%{x}<br>Turns: %{y}<extra>" + label + "</extra>",
        showlegend=True,
    ), row=1, col=1)

    fig.add_trace(go.Box(
        x=sub.scenario_short, y=sub.cost_usd, name=label, legendgroup=label,
        marker_color=color, boxpoints="all", jitter=0.4, pointpos=0,
        marker_size=6, line_width=1.5,
        hovertemplate="%{x}<br>Cost: $%{y:.4f}<extra>" + label + "</extra>",
        showlegend=False,
    ), row=1, col=2)

fig.update_layout(**LAYOUT, height=520, width=1100, boxmode="group",
    title=dict(text="Session Variability", y=0.98, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=120, b=50, l=60, r=30), legend=LEGEND_TOP)
fig.update_yaxes(title_text="Turns", row=1, col=1)
fig.update_yaxes(title_text="USD", tickprefix="$", row=1, col=2)
fig.show()

## 5. Token Breakdown

In [157]:
token_cols = ["total_input", "total_cache_read", "total_cache_creation", "total_output"]
token_labels = {"total_input": "New input", "total_cache_read": "Cache read (75% off)",
                "total_cache_creation": "Cache creation", "total_output": "Output"}
token_colors = {"New input": "#2563EB", "Cache read (75% off)": "#10B981",
                "Cache creation": "#F59E0B", "Output": "#EF4444"}

if all(c in df.columns for c in token_cols):
    scenarios = sorted(df.scenario_num.unique())
    n = len(scenarios)
    fig = make_subplots(rows=2, cols=4,
        subplot_titles=[f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in scenarios],
        horizontal_spacing=0.06, vertical_spacing=0.15)

    # Short legend labels for each condition
    short_labels = {l: l.replace("Notion Agent CLI", "NAC").replace("Notion MCP", "MCP") for l in labels}

    for idx, s in enumerate(scenarios):
        row, col = idx // 4 + 1, idx % 4 + 1
        sub = df[df.scenario_num == s]

        for comp_col, comp_name in token_labels.items():
            for li, label in enumerate(labels):
                lsub = sub[sub.label == label]
                if lsub.empty: continue
                val = lsub[comp_col].mean()
                fig.add_trace(go.Bar(
                    name=comp_name, x=[short_labels[label]], y=[val],
                    marker_color=token_colors[comp_name],
                    showlegend=(idx == 0 and li == 0),
                    legendgroup=comp_name,
                    hovertemplate=f"{short_labels[label]}<br>{comp_name}: %{{y:,.0f}}<extra>S{s}</extra>",
                ), row=row, col=col)

    fig.update_layout(**LAYOUT, barmode="stack", height=700, width=1100,
        title=dict(text="Token Breakdown per Scenario", y=0.98, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=100, b=40, l=60, r=30),
        legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0.5, xanchor="center"))
    fig.update_yaxes(title_text="Tokens", row=1, col=1)
    fig.update_yaxes(title_text="Tokens", row=2, col=1)
    fig.show()
else:
    print("Token data not available")

## 6. Savings

In [158]:
if len(labels) >= 2:
    la, lb = labels[0], labels[1]
    rows = []
    for s in sorted(df.scenario_num.unique()):
        sa = df[(df.scenario_num == s) & (df.label == la)]
        sb = df[(df.scenario_num == s) & (df.label == lb)]
        if sa.empty or sb.empty: continue
        at, bt = sa.num_turns.mean(), sb.num_turns.mean()
        ac, bc = sa.cost_usd.mean(), sb.cost_usd.mean()
        rows.append({"scenario_num": s,
            "scenario": f"S{s} {SCENARIO_NAMES.get(s, '')}",
            "turn_saving": (1 - at / bt) * 100 if bt else 0,
            "cost_saving": (1 - ac / bc) * 100 if bc else 0})

    sdf = pd.DataFrame(rows)

    fig = make_subplots(rows=1, cols=2, subplot_titles=["Turn Reduction %", "Cost Reduction %"],
                        horizontal_spacing=0.18, shared_yaxes=True)

    for col_idx, metric in enumerate(["turn_saving", "cost_saving"], 1):
        colors = ["#10B981" if v > 0 else "#EF4444" for v in sdf[metric]]
        fig.add_trace(go.Bar(
            y=sdf.scenario, x=sdf[metric], orientation="h",
            marker_color=colors, showlegend=False,
            text=sdf[metric].apply(lambda v: f"{v:+.0f}%"),
            textposition="outside", textfont_size=12,
        ), row=1, col=col_idx)
        fig.add_vline(x=0, line_dash="dash", line_color="#ccc", line_width=1, row=1, col=col_idx)

    a_t = df[df.label == la].cost_usd.sum()
    b_t = df[df.label == lb].cost_usd.sum()
    total_pct = (1 - a_t / b_t) * 100 if b_t else 0
    fig.add_annotation(x=0.98, y=-0.08, xref="paper", yref="paper",
        text=f"<b>Total: {la} ${a_t:.2f} vs {lb} ${b_t:.2f} ({total_pct:+.0f}%)</b>",
        showarrow=False, font=dict(size=12), xanchor="right")

    fig.update_layout(**LAYOUT, showlegend=False, height=440, width=1000,
        title=dict(text=f"Savings — {la} vs {lb} (positive = {la} wins)",
                   y=0.97, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=70, b=50, l=140, r=70))
    fig.update_xaxes(title_text="%", range=[0, 108], row=1, col=1)
    fig.update_xaxes(title_text="%", range=[0, 108], row=1, col=2)
    fig.show()

## 7. Cumulative Cost per Turn

Shows how billed input tokens accumulate as turns progress. Each turn re-sends the full conversation context, so more turns = quadratic cost growth. NAC finishes in 2-3 turns; MCP needs 4-40.

In [159]:
showcase = [4, 2, 6]
available = [s for s in showcase if s in df.scenario_num.values]
if not available:
    available = sorted(df.scenario_num.unique())[:3]

fig = make_subplots(rows=1, cols=len(available),
                    subplot_titles=[f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in available],
                    horizontal_spacing=0.12)

for col, s in enumerate(available, 1):
    max_turns = 0
    for label in labels:
        sub = df[(df.scenario_num == s) & (df.label == label) & (df.iteration == 1)]
        if sub.empty:
            continue
        turn_tokens = sub.iloc[0].get("turn_tokens", [])
        if not turn_tokens:
            continue
        billed = [t["billed"] for t in turn_tokens]
        cumulative = np.cumsum(billed)
        max_turns = max(max_turns, len(cumulative))
        color = PALETTE.get(label, "#888")

        fig.add_trace(go.Scatter(
            x=list(range(1, len(cumulative) + 1)), y=cumulative,
            mode="lines+markers", name=label, legendgroup=label,
            line=dict(color=color, width=2.5), marker=dict(size=6),
            showlegend=(col == 1),
            hovertemplate=f"Turn %{{x}}<br>Cumulative: %{{y:,.0f}} tokens<extra>{label}</extra>",
        ), row=1, col=col)

    tick_step = 1 if max_turns <= 8 else 5
    fig.update_xaxes(title_text="Turn", dtick=tick_step, row=1, col=col)

fig.update_yaxes(title_text="Cumulative billed input tokens", row=1, col=1)
fig.update_layout(**LAYOUT, height=480, width=1100,
    title=dict(text="Context Growth per Turn (billed input tokens)",
               y=0.99, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=110, b=60, l=80, r=30),
    legend=dict(orientation="h", yanchor="top", y=1.15, x=0.5, xanchor="center"))
fig.show()

## 8. Turns vs Cost — Efficiency Frontier

Each dot is one session. Reveals the structural relationship: more turns = more cost (context replay). NAC clusters in the low-turn/low-cost corner; MCP spreads along the cost axis.

In [160]:
fig = go.Figure()

for label in labels:
    sub = df[df.label == label]
    color = PALETTE.get(label, "#888")

    fig.add_trace(go.Scatter(
        x=sub.num_turns, y=sub.cost_usd, mode="markers",
        name=label, marker=dict(color=color, size=10, opacity=0.8),
        hovertemplate="%{text}<br>Turns: %{x}<br>Cost: $%{y:.4f}<extra>" + label + "</extra>",
        text=sub.scenario,
    ))

    if len(sub) > 2:
        z = np.polyfit(sub.num_turns, sub.cost_usd, 2)
        p = np.poly1d(z)
        x_range = np.linspace(sub.num_turns.min(), sub.num_turns.max(), 50)
        fig.add_trace(go.Scatter(
            x=x_range, y=p(x_range), mode="lines",
            line=dict(color=color, width=1.5, dash="dot"),
            showlegend=False, hoverinfo="skip",
        ))

fig.update_layout(**LAYOUT, height=500, width=850,
    title=dict(text="Turns vs Cost: Each Dot is One Session",
               y=0.97, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=80, b=60, l=70, r=30),
    yaxis_tickprefix="$", xaxis_title="Agentic Turns", yaxis_title="Cost (USD)",
    legend=LEGEND_H)
fig.show()

## 9. Output Token Ratio — I/O Format Impact

Compares total output tokens (what Claude generates) per scenario. MCP requires Claude to construct verbose JSON block structures; NAC accepts markdown, producing smaller output. This compounds via context replay on subsequent turns.

In [161]:
if "total_output" in df.columns and "total_input" in df.columns:
    io_agg = df.groupby(["scenario_num", "scenario_short", "label"]).agg(
        input_mean=("total_input", "mean"),
        output_mean=("total_output", "mean"),
    ).reset_index()
    io_agg["output_ratio"] = io_agg["output_mean"] / (io_agg["input_mean"] + io_agg["output_mean"]) * 100

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["Total Output Tokens (avg)", "Output / Total Ratio"],
                        horizontal_spacing=0.14)

    for label in labels:
        sub = io_agg[io_agg.label == label].sort_values("scenario_num")
        color = PALETTE.get(label, "#888")

        fig.add_trace(go.Bar(
            name=label, x=sub.scenario_short, y=sub.output_mean,
            marker_color=color, opacity=0.85, legendgroup=label, showlegend=True,
            hovertemplate="%{x}<br>Output: %{y:,.0f} tokens<extra>" + label + "</extra>",
        ), row=1, col=1)

        fig.add_trace(go.Bar(
            name=label, x=sub.scenario_short, y=sub.output_ratio,
            marker_color=color, opacity=0.85, legendgroup=label, showlegend=False,
            hovertemplate="%{x}<br>Output ratio: %{y:.1f}%<extra>" + label + "</extra>",
        ), row=1, col=2)

    fig.update_yaxes(title_text="Tokens", row=1, col=1)
    fig.update_yaxes(title_text="%", row=1, col=2)
    fig.update_layout(**LAYOUT, barmode="group", height=520, width=1100,
        title=dict(text="Output Verbosity: I/O Format Impact", y=0.98, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=120, b=50, l=60, r=30), legend=LEGEND_TOP)
    fig.show()
else:
    print("Token data not available")

## 10. Export to assets/

Re-run this cell to overwrite the PNGs used by README.md. Requires `kaleido` (`pip install kaleido`).

In [162]:
ASSETS_DIR.mkdir(exist_ok=True)
SCALE = 2
charts = {}
short_labels = {l: l.replace("Notion Agent CLI", "NAC").replace("Notion MCP", "MCP") for l in labels}

# ── Helper: build the main cost figure (shared between interactive + export) ──
def _build_cost_figure():
    pairs = _model_pairs(labels)
    agg = df.groupby(["scenario_num", "label"]).agg(
        cost_mean=("cost_usd", "mean"), cost_std=("cost_usd", "std"),
        n=("cost_usd", "count")).reset_index()
    slabels = [f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in sorted(agg.scenario_num.unique())]
    x_max = (agg.cost_mean + agg.cost_std.fillna(0)).max() * 1.30

    fig = make_subplots(rows=1, cols=len(pairs),
        subplot_titles=["Sonnet 4.6", "Opus 4.6"][:len(pairs)],
        horizontal_spacing=0.20, shared_yaxes=True)
    for col, (model, cli_label, mcp_label) in enumerate(pairs, 1):
        for label, color, name in [(mcp_label, MCP_COLOR, "MCP"), (cli_label, NAC_COLOR, "NAC")]:
            sub = agg[agg.label == label].sort_values("scenario_num")
            sub_labels = sub.scenario_num.map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")
            texts = [f"${r.cost_mean:.3f}" if r.cost_mean > x_max * 0.08 else ""
                     for _, r in sub.iterrows()]
            fig.add_trace(go.Bar(y=sub_labels, x=sub.cost_mean, orientation="h",
                error_x=dict(type="data", array=sub.cost_std.fillna(0), thickness=1.5, width=3),
                name=name, legendgroup=name, showlegend=(col == 1),
                marker_color=color, opacity=0.85,
                text=texts, textposition="outside", textfont_size=10,
                hovertemplate="%{y}<br>" + name + ": $%{x:.4f} \u00b1 $%{error_x.array:.4f}<extra></extra>",
            ), row=1, col=col)
        fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(slabels)), row=1, col=col)
        fig.update_xaxes(title_text="USD", tickprefix="$", range=[0, x_max], row=1, col=col)

    n_per = int(agg.n.mode().iloc[0]) if not agg.empty else "?"
    fig.update_layout(**LAYOUT, barmode="group", height=540, width=1100,
        title=dict(text="Mean Session Cost by Scenario (\u00b1 SD)",
                   y=0.98, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=100, b=60, l=160, r=80),
        legend=dict(orientation="h", yanchor="bottom", y=1.04, x=0.5, xanchor="center"))
    fig.add_annotation(x=0.5, y=-0.08, xref="paper", yref="paper", showarrow=False,
        text=f"Error bars: mean \u00b1 1 SD, n={n_per} per condition per scenario",
        font=dict(size=11, color="#666"), xanchor="center")
    return fig

# ── Helper: build the paired dumbbell figure ──────────────────────────────────
def _build_paired_figure():
    pairs = _model_pairs(labels)
    agg_p = df.groupby(["scenario_num", "label"]).agg(
        turns_mean=("num_turns", "mean"), cost_mean=("cost_usd", "mean")).reset_index()
    slabels = [f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in sorted(agg_p.scenario_num.unique())]
    titles = [f"Turns \u2014 {m}" for m, _, _ in pairs] + [f"Cost \u2014 {m}" for m, _, _ in pairs]
    fig = make_subplots(rows=2, cols=len(pairs), subplot_titles=titles,
        horizontal_spacing=0.16, vertical_spacing=0.16)
    turns_max = float(agg_p.turns_mean.max()) * 1.20
    cost_max = float(agg_p.cost_mean.max()) * 1.20
    for col, (model, cli_label, mcp_label) in enumerate(pairs, 1):
        cli = agg_p[agg_p.label == cli_label][["scenario_num", "turns_mean", "cost_mean"]].rename(
            columns={"turns_mean": "turns_cli", "cost_mean": "cost_cli"})
        mcp = agg_p[agg_p.label == mcp_label][["scenario_num", "turns_mean", "cost_mean"]].rename(
            columns={"turns_mean": "turns_mcp", "cost_mean": "cost_mcp"})
        merged = cli.merge(mcp, on="scenario_num", how="inner").sort_values("scenario_num")
        merged["scenario_label"] = merged["scenario_num"].map(lambda n: f"S{n} {SCENARIO_NAMES.get(n, '')}")
        for row, (cli_m, mcp_m, x_title, x_mx) in enumerate([
            ("turns_cli", "turns_mcp", "Turns", turns_max),
            ("cost_cli", "cost_mcp", "USD", cost_max),
        ], 1):
            for _, item in merged.iterrows():
                lo, hi = sorted((item[cli_m], item[mcp_m]))
                fig.add_trace(go.Scatter(x=[lo, hi], y=[item["scenario_label"], item["scenario_label"]],
                    mode="lines", line=dict(color="#CBD5E1", width=3),
                    showlegend=False, hoverinfo="skip"), row=row, col=col)
            fig.add_trace(go.Scatter(x=merged[cli_m], y=merged["scenario_label"], mode="markers",
                name="NAC", legendgroup="NAC", showlegend=(col == 1 and row == 1),
                marker=dict(color=NAC_COLOR, size=11, line=dict(color="white", width=1))),
                row=row, col=col)
            fig.add_trace(go.Scatter(x=merged[mcp_m], y=merged["scenario_label"], mode="markers",
                name="MCP", legendgroup="MCP", showlegend=(col == 1 and row == 1),
                marker=dict(color=MCP_COLOR, size=11, line=dict(color="white", width=1))),
                row=row, col=col)
            fig.update_xaxes(title_text=x_title, range=[0, x_mx], row=row, col=col)
            if row == 2:
                fig.update_xaxes(tickprefix="$", row=row, col=col)
            fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(slabels)),
                showticklabels=(col == 1), row=row, col=col)
    fig.update_layout(**LAYOUT, height=760, width=1180,
        title=dict(text="Per-Scenario Comparison: NAC vs MCP by Model",
                   y=0.985, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=120, b=60, l=170, r=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0.5, xanchor="center"))
    return fig

# ── Build and export all charts ───────────────────────────────────────────────
charts["bench-turns-cost.png"] = _build_cost_figure()
charts["bench-turns-cost-paired.png"] = _build_paired_figure()

# Savings
if len(labels) >= 2:
    la, lb = labels[0], labels[1]
    rows = []
    for s in sorted(df.scenario_num.unique()):
        sa = df[(df.scenario_num == s) & (df.label == la)]
        sb = df[(df.scenario_num == s) & (df.label == lb)]
        if sa.empty or sb.empty: continue
        at, bt = sa.num_turns.mean(), sb.num_turns.mean()
        ac, bc = sa.cost_usd.mean(), sb.cost_usd.mean()
        rows.append({"scenario": f"S{s} {SCENARIO_NAMES.get(s, '')}",
            "turn_saving": (1 - at / bt) * 100 if bt else 0,
            "cost_saving": (1 - ac / bc) * 100 if bc else 0})
    sdf = pd.DataFrame(rows)
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Turn Reduction %", "Cost Reduction %"],
                        horizontal_spacing=0.18, shared_yaxes=True)
    for col_idx, metric in enumerate(["turn_saving", "cost_saving"], 1):
        colors = ["#10B981" if v > 0 else "#EF4444" for v in sdf[metric]]
        fig.add_trace(go.Bar(y=sdf.scenario, x=sdf[metric], orientation="h",
            marker_color=colors, showlegend=False,
            text=sdf[metric].apply(lambda v: f"{v:+.0f}%"), textposition="outside", textfont_size=12,
        ), row=1, col=col_idx)
        fig.add_vline(x=0, line_dash="dash", line_color="#ccc", line_width=1, row=1, col=col_idx)
    a_t = df[df.label == la].cost_usd.sum()
    b_t = df[df.label == lb].cost_usd.sum()
    total_pct = (1 - a_t / b_t) * 100 if b_t else 0
    fig.add_annotation(x=0.98, y=-0.08, xref="paper", yref="paper",
        text=f"<b>Total: {la} ${a_t:.2f} vs {lb} ${b_t:.2f} ({total_pct:+.0f}%)</b>",
        showarrow=False, font=dict(size=12), xanchor="right")
    fig.update_layout(**LAYOUT, showlegend=False, height=440, width=1000,
        title=dict(text=f"Savings: {la} vs {lb}", y=0.97, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=70, b=50, l=140, r=70))
    fig.update_xaxes(title_text="%", range=[0, 108], row=1, col=1)
    fig.update_xaxes(title_text="%", range=[0, 108], row=1, col=2)
    charts["bench-savings.png"] = fig

# Token Breakdown per Scenario
token_cols = ["total_input", "total_cache_read", "total_cache_creation", "total_output"]
_tlabels = {"total_input": "New input", "total_cache_read": "Cache read (75% off)",
            "total_cache_creation": "Cache creation", "total_output": "Output"}
_tcolors = {"New input": "#2563EB", "Cache read (75% off)": "#10B981",
            "Cache creation": "#F59E0B", "Output": "#EF4444"}
if all(c in df.columns for c in token_cols):
    scenarios = sorted(df.scenario_num.unique())
    fig = make_subplots(rows=2, cols=4,
        subplot_titles=[f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in scenarios],
        horizontal_spacing=0.06, vertical_spacing=0.15)
    for idx, s in enumerate(scenarios):
        row, col = idx // 4 + 1, idx % 4 + 1
        sub = df[df.scenario_num == s]
        for comp_col, comp_name in _tlabels.items():
            for li, label in enumerate(labels):
                lsub = sub[sub.label == label]
                if lsub.empty: continue
                fig.add_trace(go.Bar(name=comp_name, x=[short_labels[label]], y=[lsub[comp_col].mean()],
                    marker_color=_tcolors[comp_name], showlegend=(idx == 0 and li == 0),
                    legendgroup=comp_name), row=row, col=col)
    fig.update_layout(**LAYOUT, barmode="stack", height=700, width=1100,
        title=dict(text="Token Breakdown per Scenario", y=0.98, x=0.5, xanchor="center", font_size=16),
        margin=dict(t=100, b=40, l=60, r=30),
        legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0.5, xanchor="center"))
    fig.update_yaxes(title_text="Tokens", row=1, col=1)
    fig.update_yaxes(title_text="Tokens", row=2, col=1)
    charts["bench-token-breakdown.png"] = fig

# Scatter
fig = go.Figure()
for label in labels:
    sub = df[df.label == label]
    color = PALETTE.get(label, "#888")
    fig.add_trace(go.Scatter(x=sub.num_turns, y=sub.cost_usd, mode="markers",
        name=label, marker=dict(color=color, size=10, opacity=0.8), text=sub.scenario))
    if len(sub) > 2:
        z = np.polyfit(sub.num_turns, sub.cost_usd, 2)
        p = np.poly1d(z)
        x_range = np.linspace(sub.num_turns.min(), sub.num_turns.max(), 50)
        fig.add_trace(go.Scatter(x=x_range, y=p(x_range), mode="lines",
            line=dict(color=color, width=1.5, dash="dot"), showlegend=False, hoverinfo="skip"))
fig.update_layout(**LAYOUT, height=500, width=850,
    title=dict(text="Turns vs Cost: Each Dot is One Session", y=0.97, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=100, b=60, l=70, r=30),
    yaxis_tickprefix="$", xaxis_title="Agentic Turns", yaxis_title="Cost (USD)", legend=LEGEND_TOP)
charts["bench-turns-vs-cost.png"] = fig

# Cumulative
showcase = [4, 2, 6]
available = [s for s in showcase if s in df.scenario_num.values]
fig = make_subplots(rows=1, cols=len(available),
    subplot_titles=[f"S{s} {SCENARIO_NAMES.get(s, '')}" for s in available], horizontal_spacing=0.12)
for col, s in enumerate(available, 1):
    max_turns = 0
    for label in labels:
        sub = df[(df.scenario_num == s) & (df.label == label) & (df.iteration == 1)]
        if sub.empty: continue
        turn_tokens = sub.iloc[0].get("turn_tokens", [])
        if not turn_tokens: continue
        billed = [t["billed"] for t in turn_tokens]
        cumulative = np.cumsum(billed)
        max_turns = max(max_turns, len(cumulative))
        color = PALETTE.get(label, "#888")
        fig.add_trace(go.Scatter(x=list(range(1, len(cumulative) + 1)), y=cumulative,
            mode="lines+markers", name=label, legendgroup=label,
            line=dict(color=color, width=2.5), marker=dict(size=6),
            showlegend=(col == 1)), row=1, col=col)
    fig.update_xaxes(title_text="Turn", dtick=1 if max_turns <= 8 else 5, row=1, col=col)
fig.update_yaxes(title_text="Cumulative billed input tokens", row=1, col=1)
fig.update_layout(**LAYOUT, height=480, width=1100,
    title=dict(text="Context Growth per Turn (billed input tokens)", y=0.99, x=0.5, xanchor="center", font_size=16),
    margin=dict(t=130, b=60, l=80, r=30),
    legend=dict(orientation="h", yanchor="top", y=1.18, x=0.5, xanchor="center"))
charts["bench-cumulative.png"] = fig

# Write all
for name, fig in charts.items():
    out = ASSETS_DIR / name
    fig.write_image(str(out), scale=SCALE)
    print(f"  -> {out}")

print(f"\nExported {len(charts)} charts to {ASSETS_DIR}")

  -> assets/bench-turns-cost.png
  -> assets/bench-turns-cost-paired.png
  -> assets/bench-savings.png
  -> assets/bench-token-breakdown.png
  -> assets/bench-turns-vs-cost.png
  -> assets/bench-cumulative.png

Exported 6 charts to assets
